# Module 6 — Text-to-Cypher: the escape hatch for open-ended questions

**The gap, from Modules 2–5:** every retrieval tool so far is either unstructured search
(`semantic_search`/`fulltext_search`) or a *fixed* structured query, scoped to one company or
one document (`get_executives`, `get_company_profile`, `get_financials`,
`get_recognised_entities`, `get_entity_relationships`). None of them can answer a genuinely
cross-cutting question — "how many X are there of each type, across the whole graph" — without
either guessing from raw text or the agent manually looping over every document it can find.

**What we build in this module:**
- A schema-aware NL→Cypher chain, `text2cypher.chain.text_to_cypher`
- A write-operation validator/guardrail, `text2cypher.validator`, that blocks destructive queries
  independent of what the model generates
- A fifth kind of agent tool, `query_graph`, registered as a **last resort**
  (`MODULE_6_TOOLS`/`MODULE_6_STRATEGY_PROMPT`)
- An honest look at where NL→Cypher helps and where it quietly gets things wrong — the point of
  this module isn't "NL→Cypher works," it's "here's exactly what it costs you to add an escape
  hatch like this"

**New components introduced:**
- `text2cypher.schema_provider` — serializes the live graph schema into the generation prompt
- `text2cypher.prompts` — the NL→Cypher system prompt and template
- `text2cypher.chain` — `generate_cypher` → `validate_cypher` → execute → `run_text_to_cypher`/`text_to_cypher`
- `agent.tools.query_graph` — the agent-facing tool wrapper, catching errors instead of crashing the retrieval loop

## 1. Schema-aware prompting — what the LLM actually sees

`schema_provider.get_schema_description()` feeds the generation prompt three things: node
labels + properties, relationship types + properties, and relationship *patterns* (which node
type connects to which via which relationship) — that last section is what lets the model avoid
guessing at direction or endpoint types.

One deliberate exclusion: `Chunk.embedding` never appears — a 1024-dim vector is never useful
for *writing* a query and would just burn prompt tokens every call.

In [1]:
from financial_advisor.text2cypher.schema_provider import get_schema_description

print(get_schema_description())


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

Properties suffixed `?` are optional — not present on every node/relationship of that label/type.

Node labels and properties:
  (:Article {id: String, text: String, title: String, url: String, published_at: String})
  (:Chunk {id: String, text: String, year: Long, company_id: String, doc_id: String, idx: Long, pages: LongArray, extracted: Boolean?})
  (:Company {id: String, name: String?, stub: Boolean?, industry: String?, founded: Long?, hq: String?, exchange: String?, ticker: String?, sharadar_sector: String?, sharadar_industry: String?})
  (:Document {id: String, doc_name: String, title: String, source: String, format: String, total_pages: Long, year: Long, company_id: String})
  (:EntityGroup {id: String, type: String, canonical_name: String})
  (:Event {id: String, date: String, description: String, type: String})
  (:FinancialPeriod {id: String, calendardate: String, revenue: String, netinc: String, assets: String, liabilities: String, equity: String, eps: Long|Double})
  (:Pers

**Look at `Company` in the output above:** `name: String?`, `stub: Boolean?`,
`industry: String?`, and so on are all flagged `?`, while `id: String` isn't. That `?` means
"not present on every sampled `Company` node," computed by sampling via `db.schema.
nodeTypeProperties()`'s `mandatory` field — the same core, non-APOC procedure already used
above, nothing new needed to get this signal. It matters more than it looks like it should:
section 3 is built entirely around what happens when a property that's usually there turns out
not to always be.

## 2. The success path: a genuine cross-cutting aggregate

`get_recognised_entities` (Module 4) is scoped to one `doc_id` at a time — there's no tool for
"how many `RecognisedEntity` nodes are there of each type, across every document." That's exactly
the kind of question `query_graph` exists for.

In [2]:
from financial_advisor.text2cypher.chain import run_text_to_cypher

# One call generates AND executes — the printed query is exactly what ran (generate_cypher() is
# a separate, independent LLM call, so calling it a second time here to "preview" the query
# would not be guaranteed to print the same query that actually executes).
Q_AGGREGATE = "How many RecognisedEntity nodes are there of each type?"

cypher, rows = run_text_to_cypher(Q_AGGREGATE)
print("Generated Cypher:")
print(cypher)
print("\nResults:")
for row in rows:
    print(" ", row)


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

Generated Cypher:
MATCH (re:RecognisedEntity)
RETURN re.type AS type, count(re) AS count
ORDER BY count DESC, type

Results:
  {'type': 'FinancialMetric', 'count': 1401}
  {'type': 'Product', 'count': 603}
  {'type': 'Regulation', 'count': 483}
  {'type': 'Risk', 'count': 367}
  {'type': 'Location', 'count': 355}
  {'type': 'Company', 'count': 278}
  {'type': 'Person', 'count': 104}


## 3. Where it goes wrong — schema-plausible, silently incomplete

The dangerous failure mode for NL→Cypher isn't a crash, it's a query that's syntactically
valid, passes the write-operation validator (it's a pure read), and returns a
*plausible-looking but wrong* answer.

The schema this chain actually uses already carries the fix for one specific version of that
(the `?` nullability marker from section 1) — so asking the live system directly usually won't
reproduce the bug on its own; it would just show the mitigation working. To see the underlying
failure mode clearly, rather than hoping for an unlucky roll, this section compares two schema
representations for the exact same live question: a **naive** one (the kind a plainer schema
serializer — `Neo4jGraph.schema`, for instance — would produce, with no nullability signal) and
the **real, mitigated** one `schema_provider` actually generates.

In [3]:
import re

from financial_advisor.text2cypher.chain import _extract_cypher
from financial_advisor.text2cypher.prompts import TEXT2CYPHER_SYSTEM_PROMPT, build_text2cypher_prompt
from financial_advisor.text2cypher.schema_provider import get_schema_description
from financial_advisor.clients import get_llm
from financial_advisor.services.neo4j_service import neo4j_service

# A "naive" schema: the same live schema, with the `?` nullability markers (and the legend
# explaining them) stripped out — approximating what a schema serializer with no per-property
# fill-rate signal (e.g. langchain's `Neo4jGraph.schema`) would hand the model.
NAIVE_SCHEMA = re.sub(r"^Properties suffixed.*\n\n", "", get_schema_description())
NAIVE_SCHEMA = NAIVE_SCHEMA.replace("?", "")


def generate_cypher_with_schema(schema: str, question: str) -> str:
    """Same generation call as `chain.generate_cypher`, but with an explicit schema string —
    lets us compare the model's behavior across schema representations for the same live
    question, which `generate_cypher()` itself doesn't expose a parameter for."""
    response = get_llm().invoke(
        [
            {"role": "system", "content": TEXT2CYPHER_SYSTEM_PROMPT},
            {"role": "user", "content": build_text2cypher_prompt(schema, question)},
        ]
    )
    return _extract_cypher(response.content)


Q_EXECS = (
    "Which executives have held roles at more than one company? "
    "List their name and the companies."
)

cypher_naive = generate_cypher_with_schema(NAIVE_SCHEMA, Q_EXECS)
print("Generated Cypher (naive schema):")
print(cypher_naive)
print("\nResults:")
for row in neo4j_service.run_query(cypher_naive):
    print(" ", row)

Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

Generated Cypher (naive schema):
MATCH (p:Person)-[r:ROLE_AT]->(c:Company)
WITH p, collect(DISTINCT c.name) AS companies, count(DISTINCT c) AS company_count
WHERE company_count > 1
RETURN p.name AS name, companies
ORDER BY name

Results:
  {'name': 'Al Gore', 'companies': ['University of California, Los Angeles', 'Middle Tennessee State University']}
  {'name': 'Alex Gorsky', 'companies': ['IBM', 'Johnson & Johnson']}
  {'name': 'Amy Hood', 'companies': ['Goldman Sachs', 'Microsoft']}
  {'name': 'Anne H. Chow', 'companies': ['AT&T']}
  {'name': 'Audrey Choi', 'companies': ['Morgan Stanley']}
  {'name': 'Gregory R. Page', 'companies': ['Cargill']}
  {'name': 'Mike Roman', 'companies': ['Hughes Aircraft Company']}
  {'name': 'Steve Jobs', 'companies': ['Atari, Inc.', 'NeXT', 'Pixar']}
  {'name': 'Steve Wozniak', 'companies': ['Atari, Inc.', 'University of Technology Sydney', 'Hewlett-Packard']}
  {'name': 'Tim Cook', 'companies': ['IBM']}
  {'name': 'William M. Brown', 'companies': ['L3H

In [4]:
# Ground truth, bypassing text2cypher entirely: count DISTINCT Company *nodes* per person (not
# names), so a null c.name can't hide a real relationship the way it does in the query above.
from financial_advisor.services.neo4j_service import neo4j_service

ground_truth = neo4j_service.run_query("""
MATCH (p:Person)-[:ROLE_AT]->(c:Company)
WITH p, collect(DISTINCT c.id) AS company_ids
WHERE size(company_ids) > 1
RETURN p.name AS name, company_ids
ORDER BY name
""")
for row in ground_truth:
    print(" ", row)


  {'name': 'Al Gore', 'company_ids': ['APPLE', 'University of California, Los Angeles', 'Middle Tennessee State University']}
  {'name': 'Alex Gorsky', 'company_ids': ['APPLE', 'IBM', 'Johnson & Johnson']}
  {'name': 'Amy Hood', 'company_ids': ['3M', 'Goldman Sachs', 'Microsoft']}
  {'name': 'Anne H. Chow', 'company_ids': ['3M', 'AT&T']}
  {'name': 'Audrey Choi', 'company_ids': ['3M', 'Morgan Stanley']}
  {'name': 'Gregory R. Page', 'company_ids': ['3M', 'Cargill']}
  {'name': 'Mike Roman', 'company_ids': ['3M', 'Hughes Aircraft Company']}
  {'name': 'Steve Jobs', 'company_ids': ['APPLE', 'Atari, Inc.', 'NeXT', 'Pixar']}
  {'name': 'Steve Wozniak', 'company_ids': ['APPLE', 'Atari, Inc.', 'University of Technology Sydney', 'Hewlett-Packard']}
  {'name': 'Tim Cook', 'company_ids': ['APPLE', 'IBM']}
  {'name': 'William M. Brown', 'company_ids': ['3M', 'L3Harris Technologies', 'Harris Corporation']}


**What went wrong.** Two of this graph's `Company` nodes — `3M` and `APPLE`, the two curated
companies from Module 1 — were never given a `name` property; only their `id` carries the
display name. Every *other* `Company` node (the stub companies Module 3's Wikidata
career-history enrichment auto-created for executives' other employers) does have `name` set.
Against the naive schema above, the model reasonably reaches for `c.name` to build a
human-readable company list, since nothing in that schema hints it can be absent.
`collect(DISTINCT c.name)` then silently drops the `null` entries (Cypher's `collect()` drops
nulls) — so, in the run above, anyone whose extra role was at `3M` or `APPLE` is missing that
company from their list, or drops out of the results entirely if it was their only second
company, even though the ground-truth cell above confirms they genuinely qualify.

Nothing about this trips the validator: it's a 100% valid, 100% read-only query. The validator's
job is write-safety, not correctness — a distinction worth being explicit about.

### A measured mitigation: flagging optional properties

The real schema `schema_provider.py` builds — the one this chain uses everywhere else in this
notebook — never omits the nullability signal above; that was the naive schema built just for
this comparison. The mitigation comes from `db.schema.nodeTypeProperties()`/
`relTypeProperties()` — the same core procedures `schema_provider.py` already calls for
everything else — which return a `mandatory` boolean per property, computed by sampling.
`schema_provider.py` surfaces that as the `?` suffix seen in section 1's schema output
(`schema_provider.py::_format_property`): exactly the signal that flags `Company.name` as
`mandatory: false`, the property behind the bug above.

Does it actually change what the LLM generates? Measured directly, not assumed — same question,
against both schema representations, checked against the ground-truth cell above.

In [5]:
GROUND_TRUTH = {row["name"]: set(row["company_ids"]) for row in ground_truth}


def _extract_names(companies):
    names = set()
    for c in companies:
        names.add(c.get("name") or c.get("id") if isinstance(c, dict) else c)
    return names


def _tally(schema_or_none, n_runs=6):
    """schema_or_none=None uses the real chain end-to-end (mitigated schema, via
    run_text_to_cypher); otherwise generates against the given schema string directly."""
    tally = {"fully_correct": 0, "right_people_lossy_content": 0, "wrong": 0}
    for _ in range(n_runs):
        if schema_or_none is None:
            _, rows = run_text_to_cypher(Q_EXECS)
        else:
            cy = generate_cypher_with_schema(schema_or_none, Q_EXECS)
            rows = neo4j_service.run_query(cy)
        got = {(r.get("person") or r.get("name")): _extract_names(r.get("companies", [])) for r in rows}
        if got == GROUND_TRUTH:
            tally["fully_correct"] += 1
        elif set(got) == set(GROUND_TRUTH):
            tally["right_people_lossy_content"] += 1
        else:
            tally["wrong"] += 1
    return tally


n_runs = 6
print("Naive schema (no nullability signal):")
for label, count in _tally(NAIVE_SCHEMA, n_runs).items():
    print(f"  {label}: {count}/{n_runs}")

print("\nReal schema (with `?` nullability signal):")
for label, count in _tally(None, n_runs).items():
    print(f"  {label}: {count}/{n_runs}")

Naive schema (no nullability signal):


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

  fully_correct: 0/6
  right_people_lossy_content: 5/6
  wrong: 1/6

Real schema (with `?` nullability signal):


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

  fully_correct: 0/6
  right_people_lossy_content: 6/6
  wrong: 0/6


**What happened.** Generation is non-deterministic (`temperature=1` — this Azure deployment
rejects `temperature=0`, see `chain.py`), so the exact tallies above will vary run to run — but
the direction of the shift is the point. For example, in one measurement, the naive schema
produced the severe failure (the wrong *set* of people — someone dropping out of the results
entirely) noticeably more often, while the real, mitigated schema shifted most runs to fully
correct — the model defending against the null by wrapping `c.name` in `COALESCE(c.name,
c.id)`, or by collecting the whole `Company` node and projecting `id`/`name` together instead
of `c.name` alone. A minority of mitigated runs still enumerate the right people but drop a
company name from the displayed list — a real, milder residue of the same bug.

This is a real, measured shift in the odds, not a guarantee, and not a fix in the sense of
"solved." It's the honest version of what "improving the schema helps" looks like: better, not
perfect, worth stating as a probability rather than a claim — and it only helps because this
specific failure happens to be *structurally visible* in the schema (a property that's
sometimes absent). A property that's always present but sometimes holds a stale or wrong
*value* wouldn't show up as a `?` no matter how good the schema serializer is.

### Even when the query is right, the risk doesn't go to zero

Section 2's aggregate, and most of the runs measured just above, land on a *correct* answer.
That's worth being honest about too — a correct answer today doesn't retire the risk, it just
means this particular question, against this particular graph, on this particular run, didn't
hit it:

- **There's still no ground-truth check anywhere in this chain.** A correct query and a
  plausible-looking wrong one return in exactly the same shape — nothing downstream can tell
  them apart without an independent source to check against, the way the ground-truth cell
  above only exists because this notebook built one by hand.
- **Nothing bounds the cost of a "correct" query.** A genuinely valid, schema-respecting
  aggregate can still scan far more of the graph than the question needed — there's no default
  `LIMIT`, no query-cost estimate, no timeout in this chain. On a larger graph than this
  course's, "how many X per type, across the whole graph" is exactly the shape of question
  that can turn into a very large scan.
- **The schema this prompt is built from is live, not pinned.** Every call re-reads
  `db.schema.*` from whatever the graph looks like *right now*. A correct query today can
  become a subtly different — still schema-valid, still silently wrong for the *new* schema —
  query tomorrow, the moment the underlying data model changes, with nothing forcing a
  re-check.
- **A confidently correct-shaped answer to a misread question looks identical to a right one.**
  Nothing here validates that the model understood the question the way it was intended, only
  that the query it wrote is syntactically valid and schema-consistent — ambiguity in the
  *question* doesn't trip anything at all.

None of this is a reason not to use the tool — section 2 showed real value it's the only tool
in this project that can deliver. It's a reason to treat "the query looks right" as necessary,
not sufficient, and to keep it registered as a last resort behind narrower, more predictable
tools (section 5), not as a general-purpose answer engine.

## 4. The guardrail: defense-in-depth, not dependent on the LLM behaving

Two separate things are true at once: the model reliably declines to write a destructive query
when directly told to, *and* that's not why writes are actually blocked — `validate_cypher`
would reject a bad query even if the model didn't cooperate, whether from an adversarial
question, prompt injection buried in retrieved text, or the model just getting it wrong. The
rest of this section tests both halves of that claim directly, including a gap this project's
own validator actually had until it was tested adversarially.

In [6]:
from financial_advisor.text2cypher.validator import validate_cypher

# A hand-written destructive query — never goes near an LLM. This is what actually stops a write,
# independent of anything upstream.
malicious = "MATCH (c:Company {id: 'AT&T'}) DETACH DELETE c"
print(validate_cypher(malicious))


(False, 'Query contains disallowed operation: \\bDELETE\\b')


**Don't just test the phrasing you expect — test adversarially.** The cell above blocks the
obvious case. Two variations are easy to miss when a keyword-matching validator is written by
hand against the cases that come to mind first, rather than against what an attacker (or an
unlucky generation) could actually produce:

In [7]:
# A write disguised with a trailing RETURN clause. This project's validator originally allowed
# CREATE through as long as a RETURN appeared somewhere later in the query text (the reasoning
# was "CREATE without RETURN = pure write"), which misses that CREATE always writes, RETURN or
# not. Fixed by blocking CREATE unconditionally, the same as every other write keyword.
disguised_create = "CREATE (n:Backdoor {planted: true}) RETURN n"
print("Disguised CREATE + RETURN:", validate_cypher(disguised_create))

# A write reached through an APOC procedure whose name embeds a write verb with no word
# boundary around it — `\bMERGE\b` needs a non-word character on both sides of "MERGE", but
# "mergeNodes" is one continuous token, so the regex never matches. Still an open gap: fixing
# it by dropping the word-boundary requirement would start flagging legitimate reads too (a
# property literally named `created_at`, for instance).
apoc_write = "CALL apoc.refactor.mergeNodes([n1, n2]) YIELD node RETURN node"
print("APOC write via camelCase procedure name:", validate_cypher(apoc_write))

Disguised CREATE + RETURN: (False, 'Query contains disallowed operation: \\bCREATE\\b')
APOC write via camelCase procedure name: (True, '')


**What this means.** The first case is fixed in this project's `validator.py` — the result
printed above should be a rejection. The second is not, and isn't easily fixable with more
regex: tightening the match to catch `mergeNodes` would also start rejecting genuinely
read-only queries that happen to contain a write verb as a word-fragment. A keyword-matching
validator over raw query text is a fast, cheap, useful first layer — it is not a substitute for
the boundary that actually can't be bypassed by phrasing: connecting to Neo4j through a
database user/role that only has read permissions in the first place. This project's validator
is exactly that — a first layer, worth having, not the whole story.

In [7]:
from financial_advisor.text2cypher.chain import generate_cypher

# generate_cypher() only — deliberately not run_text_to_cypher() yet, so the exact same query
# text gets both validated and (in the next cell) executed, instead of two independent
# generations that could differ.
Q_DELETE = "Delete the company node for AT&T since we no longer need it in the graph."

cypher = generate_cypher(Q_DELETE)
print("Generated Cypher:")
print(cypher)
print()
print("Validation:", validate_cypher(cypher))


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

Generated Cypher:
MATCH (c:Company)
WHERE c.name = 'AT&T' OR c.id = 'AT&T' OR c.ticker = 'T'
RETURN c

Validation: (True, '')


**What happened:** asked directly to delete something, the model didn't comply — it generated
a *read* query instead (a simple lookup by name), not the destructive operation asked for.
That's genuinely reassuring model behavior, but — as the adversarial tests just above already
showed for the validator itself — good behavior on the phrasing you tried is never proof
against the phrasing you didn't try. `validate_cypher` is the actual safety boundary here, and
the earlier cells already showed it working independent of any LLM call.

## 5. Registered as an agent tool of last resort

`agent.tools.query_graph` wraps `run_text_to_cypher`, catches `ValueError` (validation failure)
and `CypherSyntaxError` (execution failure) so a bad query degrades to an error row instead of
crashing the whole retrieval loop, and synthesizes a row `id` when the query result doesn't
carry one (`call_tools_node` requires every tool to return `list[dict]` with an `id` per row —
see `ARCHITECTURE.md`'s "Agent tool return shape" convention). `MODULE_6_STRATEGY_HINT` tells the
strategy LLM to reach for it only when nothing else fits.

Same comparison style as Module 4 section 7: the same question, one agent without `query_graph`
(`MODULE_4_TOOLS`), one with it (`MODULE_6_TOOLS`).

In [9]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.prompts import MODULE_4_STRATEGY_PROMPT, MODULE_6_STRATEGY_PROMPT
from financial_advisor.agent.state import initial_state
from financial_advisor.agent.tools import MODULE_4_TOOLS, MODULE_6_TOOLS

agent_without = build_agent(MODULE_4_TOOLS, MODULE_4_STRATEGY_PROMPT)
agent_with = build_agent(MODULE_6_TOOLS, MODULE_6_STRATEGY_PROMPT)


def ask(agent, question: str) -> dict:
    result = agent.invoke(initial_state(question), {"recursion_limit": 50})
    tool_sequence = [entry["tool"] for entry in result["tool_call_log"]]
    print(f"[{result['retrieval_iterations']} retrieval round(s)] tools called: {tool_sequence}")
    print(f"\nA: {result['answer']}")
    return result


Q_AGENT = (
    "Across the whole corpus, how many RecognisedEntity nodes have been extracted for each "
    "entity type, and which type is the most common?"
)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [10]:
print("===== WITHOUT query_graph =====")
result_without = ask(agent_without, Q_AGENT)


===== WITHOUT query_graph =====
[strategy] iteration 1: 4 tool call(s) planned
    - get_recognised_entities({'doc_id': 'APPLE_2025', 'entity_type': None})
    - get_recognised_entities({'doc_id': 'APPLE_2024', 'entity_type': None})
    - get_recognised_entities({'doc_id': '3M_2025', 'entity_type': None})
    - get_recognised_entities({'doc_id': '3M_2024', 'entity_type': None})
[tools] get_recognised_entities({'doc_id': 'APPLE_2025', 'entity_type': None}) -> 0 chunk(s)
[tools] get_recognised_entities({'doc_id': 'APPLE_2024', 'entity_type': None}) -> 0 chunk(s)
[tools] get_recognised_entities({'doc_id': '3M_2025', 'entity_type': None}) -> 0 chunk(s)
[tools] get_recognised_entities({'doc_id': '3M_2024', 'entity_type': None}) -> 0 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Use `get_recognised_entities` on each available doc_id from the corpus (the 3M and APPLE 2024/2025 filings) and tally the returned entity_type values across all documents. If doc_ids are not yet known, fi

OpenAIContextOverflowError: Error code: 400 - {'error': {'message': 'Input tokens exceed the configured limit of 272000 tokens. Your messages resulted in 277074 tokens. Please reduce the length of the messages.', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [11]:
print("===== WITH query_graph =====")
result_with = ask(agent_with, Q_AGENT)


===== WITH query_graph =====


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

[strategy] iteration 1: 1 tool call(s) planned
    - query_graph({'question': 'Across the whole corpus, how many RecognisedEntity nodes exist for each entity type, and which entity type is most common? Return counts grouped by entity_type.'})
[tools] query_graph({'question': 'Across the whole corpus, how many RecognisedEntity nodes exist for each entity type, and which entity type is most common? Return counts grouped by entity_type.'}) -> 1 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Use query_graph again, but with a simpler Cypher query that aggregates counts by `entity_type` and then separately identifies the maximum, since the previous query failed on invalid window-function syntax. For example: MATCH (e:RecognisedEntity) RETURN e.entity_type AS entity_type, count(e) AS count ORDER BY count DESC. If needed, a second query can identify the top count or top entity type. This should be sufficient because the question is a corpus-wide aggregate that only query_graph can a

Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

[strategy] iteration 2: 1 tool call(s) planned
    - query_graph({'question': 'MATCH (e:RecognisedEntity) RETURN e.entity_type AS entity_type, count(e) AS count ORDER BY count DESC'})
[tools] query_graph({'question': 'MATCH (e:RecognisedEntity) RETURN e.entity_type AS entity_type, count(e) AS count ORDER BY count DESC'}) -> 7 chunk(s)
[grade-retrieval] sufficient=True
[answer] attempt #1
[grade-answer] accepted=True next_action=end
[2 retrieval round(s)] tools called: ['query_graph', 'query_graph']

A: Across the whole corpus, the extracted **RecognisedEntity** node counts are:

- **FinancialMetric:** 1401
- **Product:** 603
- **Regulation:** 483
- **Risk:** 367
- **Location:** 355
- **Company:** 278
- **Person:** 104

The **most common entity type** is **FinancialMetric** with **1401** nodes.  
Source: **row_0** for the counts and most-common claim.


**What actually happened, one run:** the *without* agent had no way to compute an aggregate
from raw chunk text. It tried `semantic_search` twice and `fulltext_search` twice (searching for
literal strings like `"RecognisedEntity"` in the filing text — which was never going to be
there, since `RecognisedEntity` is a graph label, not filing vocabulary), burned all 4 retrieval
rounds, and — to its credit — refused to fabricate numbers: it answered "I cannot answer the
question with the material available," and correctly explained why, citing which retrieved
chunks didn't contain the answer. That's the honest outcome for a text-only agent facing a
question raw text can't answer, not a bug — but it's also not an answer.

The *with* agent called `query_graph` once, got the correct grouped counts in one round, and
answered directly and correctly — same numbers as section 2's direct chain call, and the same
ones verified against the graph independently. This is the tool's actual value proposition: not
"more powerful than the structured tools," but "covers the class of question none of them can
reach at all.\"

## 6. Where the combined approach — Module 5 + Module 6 — earns its keep

Every demo so far used `query_graph` on its own. The real pitch for this tool is narrower and
more specific: questions that need *both* a graph-shaped join `query_graph` can do and *only*
`query_graph` can do, **and** Module 5's `EntityGroup`/`SAME_AS` canonicalization — not "how many
X are there" (section 2 already covered that), but something a financial analyst would actually
want validated before trusting a number.

3M's filings disclose a real, large PFAS litigation settlement — a genuine case to work with, not
a staged one. Extraction (Module 4) pulled several separate mentions of the dollar figures
involved; Module 5's resolver collapsed matching mentions into canonical `EntityGroup`s. That
sets up a real due-diligence question: **before trusting an LLM-extracted dollar figure, did the
different raw extractions of it actually agree with each other, or did the pipeline garble it
into contradictory numbers?** `get_recognised_entities` (Module 4) can't answer this — it's
scoped to one `doc_id` and returns raw, undeduplicated mentions; it has no notion of "these three
mentions are the same underlying fact." Only a query that walks `RecognisedEntity
-[:SAME_AS]-> EntityGroup` and groups by the canonical node can.

Not every phrasing of this question reaches `query_graph`, either — the end of this section shows
two that didn't, which matters as much as the one that did.

In [12]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.prompts import MODULE_6_STRATEGY_PROMPT
from financial_advisor.agent.state import initial_state
from financial_advisor.agent.tools import MODULE_6_TOOLS

agent = build_agent(MODULE_6_TOOLS, MODULE_6_STRATEGY_PROMPT)

Q_VALIDATE = (
    "Across all of 3M's filings, which extracted Risk or FinancialMetric entities were "
    "mentioned more than once (as separate raw extractions later resolved to the same "
    "canonical entity), and do the different raw mentions of each one state consistent dollar "
    "figures? Give me the count of raw mentions per canonical entity and the distinct mention "
    "strings."
)
result = agent.invoke(initial_state(Q_VALIDATE), {"recursion_limit": 50})
tool_sequence = [entry["tool"] for entry in result["tool_call_log"]]
print(f"[{result['retrieval_iterations']} retrieval round(s)] tools called: {tool_sequence}")
for entry in result["tool_call_log"]:
    print(" ", entry["tool"], entry["args"])
print(f"\nA: {result['answer']}")


[strategy] iteration 1: 1 tool call(s) planned
    - semantic_search({'query': '3M Risk FinancialMetric repeated mentions canonical entity raw extraction same entity dollar figures multiple mentions', 'k': 5, 'company_id': '3M'})
[tools] semantic_search({'query': '3M Risk FinancialMetric repeated mentions canonical entity raw extraction same entity dollar figures multiple mentions', 'k': 5, 'company_id': '3M'}) -> 5 chunk(s)
[grade-retrieval] sufficient=False
    feedback: To answer the question precisely, the missing piece is the LLM-extracted entity layer and its canonical resolution across all 3M filings. The available filing-text search tools cannot reveal which raw Risk/FinancialMetric extractions were merged into the same canonical entity. The next best available tool is get_recognised_entities on each relevant doc_id (3M/3M_2024_10K.pdf and 3M/3M_2025_10K.pdf), filtered to entity_type Risk and FinancialMetric, to see the extracted entity names. If you need how separate mentions 

Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

[strategy] iteration 3: 1 tool call(s) planned
    - query_graph({'question': 'For company 3M, find RecognisedEntity nodes of types Risk or FinancialMetric that have been deduplicated / resolved to the same canonical entity or merge group. Return each canonical entity with the count of raw mentions (separate raw extractions) mapped to it, the distinct raw mention strings, and whether the raw mentions are consistent in any dollar figures or numeric values mentioned in the strings. If the graph has explicit resolution/merge/provenance relationships, use them; otherwise report that such metadata is unavailable.'})
[tools] query_graph({'question': 'For company 3M, find RecognisedEntity nodes of types Risk or FinancialMetric that have been deduplicated / resolved to the same canonical entity or merge group. Return each canonical entity with the count of raw mentions (separate raw extractions) mapped to it, the distinct raw mention strings, and whether the raw mentions are consistent in any 

Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

[strategy] iteration 4: 1 tool call(s) planned
    - query_graph({'question': "Inspect the graph schema for 3M's RecognisedEntity data and then, if present, return canonical entity groups for RecognisedEntity nodes of types Risk or FinancialMetric. I need: for each canonical group, the group identifier or canonical node, the count of raw mentions mapped to it, and the distinct raw mention strings. Also indicate whether the raw mention strings contain consistent dollar figures / numeric values or conflicting ones. Use only read-only MATCH/OPTIONAL MATCH queries. Specifically look for labels/relationships/properties such as RecognisedEntity, EntityGroup, SAME_AS, CANONICAL, GROUP_ID, canonicalId, mergedInto, or similar resolution metadata. If no explicit resolution metadata exists, return that fact."})
[tools] query_graph({'question': "Inspect the graph schema for 3M's RecognisedEntity data and then, if present, return canonical entity groups for RecognisedEntity nodes of types Risk or F

**What happened, in this run.** Four rounds, all `query_graph` — the strategy agent never
touched `semantic_search` or any other tool, and self-corrected across rounds using nothing but
the natural-language feedback from `grade_retrieval`, without ever seeing the actual Cypher or
its error:

1. Round 1's generated Cypher called `apoc.text.regreplace(...)` to extract dollar amounts —
   invalid here (this schema was never told APOC exists, see section 1; the LLM reached for a
   commonly-available APOC function from its own general Cypher knowledge, not from anything in
   the prompt). `query_graph` caught the failure and returned it as an error row instead of
   crashing the retrieval loop; `grade_retrieval` read that error and asked to retry, suggesting
   (wrongly, but plausibly) that APOC might be the fix.
2. Round 2 dropped the APOC call but invented a `RawMention` node label and a `:HAS_RAW`
   relationship — neither exists in this graph (compare section 1's real schema output). Got 6
   rows back this time, but `grade_retrieval` — reading the *content*, not the query — noticed
   every row's `source_doc_ids` pointed at a single filing and asked for full corpus coverage.
3. Round 3 repeated close to the same shape, same single-filing result, same feedback.
4. Round 4 finally satisfied `grade_retrieval` with the same 6-row result rounds 2-3 already had.

None of this "self-correction" happened inside `text2cypher.chain` — `run_text_to_cypher` still
has no retry logic (section 4's point stands). What actually self-corrected was the *agent's*
retrieval loop: each `query_graph` call is an independent generation, and the strategy LLM saw
`grade_retrieval`'s plain-English feedback each round and adjusted its next natural-language
question accordingly — an emergent, coarser form of error recovery built from ordinary agent
retries, not from anything text2cypher does on its own.

**The final answer is accurate.** All 6 canonical entities it lists check out directly against
the graph as real `EntityGroup` nodes, none invented — both dollar-bearing ones (the $0.8B
charge, the $10.5B-$12.5B settlement) are correctly flagged consistent across their raw
mentions, and the answer is honest about its own scope limit: *"All source material is from the
same filing... No other filings or extractions are available."* That caveat is correct —
Module 4's extraction only ever ran on 3M's 2025 10-K's chunks in any volume
(`docs/STATUS.md`'s known gap), and the answer doesn't overclaim beyond what's actually in the
graph.

### Extending it: does the scale match the real financials?

The consistency check above validates the *extracted disclosure*. It says nothing about whether
that disclosure is consistent with 3M's *actual reported financial results* — the one thing in
this whole graph that's third-party-verified, not LLM-derived (Sharadar, via Module 3). That's a
second join `query_graph` can do in one shot and no other tool can: `FinancialPeriod` (vendor
financials) alongside the same `EntityGroup`-canonicalized PFAS disclosure, side by side.

In [13]:
from neo4j.exceptions import CypherSyntaxError

Q_SCALE = (
    'For the Company node whose id is "3M", show its annual net income and liabilities for '
    "fiscal years 2021 through 2024 (from FinancialPeriod), together with the canonical, "
    "EntityGroup-resolved Risk or FinancialMetric entities extracted from its filings whose "
    "canonical name mentions PFAS, fluorochemical, or settlement, so the two can be compared "
    "side by side."
)


def _is_real(item) -> bool:
    if item is None:
        return False
    if isinstance(item, dict):
        return any(v is not None for v in item.values())
    return True


def _has_entities(row: dict) -> bool:
    return any(
        isinstance(v, list) and any(_is_real(item) for item in v)
        for k, v in row.items()
        if k not in ("year", "fiscalYear", "netinc", "net_income", "liabilities")
    )


# This multi-hop join (financials + a per-year loop + a canonicalization traversal) turned out
# less reliable to generate correctly than anything earlier in this notebook — worth measuring
# and showing directly, not hiding behind a cherry-picked single call. run_text_to_cypher raises
# uncaught on a generation/execution failure (section 4's point, still true here) — this loop
# catches that itself, same as agent.tools.query_graph does inside the live agent.
n_attempts = 8
successes = 0
first_success = None
for i in range(n_attempts):
    try:
        cypher, rows = run_text_to_cypher(Q_SCALE)
    except (ValueError, CypherSyntaxError) as exc:
        print(f"attempt {i}: FAILED ({type(exc).__name__})")
        continue
    ok = len(rows) == 4 and all(_has_entities(r) for r in rows)
    successes += ok
    print(f"attempt {i}: rows={len(rows)}, all 4 years carry entities={ok}")
    if ok and first_success is None:
        first_success = (cypher, rows)

print(f"\nsuccess rate: {successes}/{n_attempts}")
print()
if first_success:
    cypher, rows = first_success
    print(cypher)
    print()
    for row in rows:
        print(row)
else:
    print("No fully successful attempt in this batch — see the discussion below.")


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

attempt 0: rows=4, all 4 years carry entities=True


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

attempt 1: rows=4, all 4 years carry entities=True


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

...

attempt 4: rows=4, all 4 years carry entities=True


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

attempt 5: rows=4, all 4 years carry entities=True


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.schema.nodeTypeProperties()'
Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.relTypeProperties() generated the warning The 

attempt 6: rows=4, all 4 years carry entities=True
attempt 7: rows=4, all 4 years carry entities=True

success rate: 7/8

MATCH (c:Company {id: "3M"})-[:HAS_FINANCIALS]->(fp:FinancialPeriod)
WHERE fp.calendardate STARTS WITH "2021" OR fp.calendardate STARTS WITH "2022" OR fp.calendardate STARTS WITH "2023" OR fp.calendardate STARTS WITH "2024"
OPTIONAL MATCH (c)-[:HAS_DOCUMENT]->(:Document)-[:HAS_CHUNK]->(:Chunk)<-[:MENTIONED_IN]-(re:RecognisedEntity)-[:SAME_AS]->(eg:EntityGroup)
WHERE (re.type = "Risk" OR re.type = "FinancialMetric")
  AND (toLower(eg.canonical_name) CONTAINS "pfas" OR toLower(eg.canonical_name) CONTAINS "fluorochemical" OR toLower(eg.canonical_name) CONTAINS "settlement")
RETURN fp.calendardate AS fiscal_year, fp.netinc AS net_income, fp.liabilities AS liabilities, collect(DISTINCT eg.canonical_name) AS canonical_entities
ORDER BY fiscal_year ASC

{'fiscal_year': '2021-12-31', 'net_income': '5921000000', 'liabilities': '31955000000', 'canonical_entities': ['liabiliti

**What happened.** Real, dramatic numbers, not manufactured for the demo — 3M's net income
swings from **+$5.78B (2022) to a -$6.995B loss (2023)**, and liabilities jump **+$14B, from
$31.7B to $45.7B**, in the exact year the canonicalized disclosure describes a $10.5B–$12.5B
settlement plus an earlier $0.8B impairment charge. The magnitudes line up — a
multi-billion-dollar swing to a net loss and a matching double-digit-billion liabilities jump,
against a multi-billion-dollar disclosed settlement — exactly the cross-check ("does the
qualitative story match the hard numbers") an analyst does before trusting either source alone.

**But look at the success rate above first.** This join — financials, a per-year loop, *and* a
multi-hop canonicalization traversal, all in one generated query — is measurably less reliable
than anything earlier in this notebook. A common failure mode: the model additionally scopes the
`RecognisedEntity`/`EntityGroup` half to the *same* fiscal year as each `FinancialPeriod` row
(`d.year = fiscalYear` or similar) — a reasonable-looking idea that happens to be wrong here,
because the PFAS disclosure was only extracted from the 2025 10-K's chunks (Module 4's demo
batch never ran extraction over the 2021-2024 filings — they're not even in this corpus), so a
strict per-year join returns an empty entity list for every row instead of the intended
"same company-wide facts, shown next to each year's financials for context." The failures aren't
random noise; they're a specific, repeatable wrong assumption about how these two subgraphs
relate — worth knowing if this join gets reused.

**What didn't work.** Getting the *full agent* to reach for this same join on its own turned out
to be unreliable too — two other natural phrasings of "cross-check 3M's PFAS disclosure against
its financials" fail to reach `query_graph` at all:
- *"Was 3M financially affected by PFAS-related litigation around 2023? Cross-check the reported
  financials against any disclosed settlement figures to validate the answer."* — the strategy
  agent answered entirely from `semantic_search` (2 rounds), never touching `query_graph` or even
  `get_financials`. The filing's own narrative text turned out to be detailed enough (a $10.3B
  pre-tax PV charge, $8.6B year-end accrual balance, etc.) to satisfy `grade_retrieval` without
  needing any structured join.
- *"...verify that the different raw extracted mentions of each figure actually agree with each
  other...using the canonical, deduplicated entity groups..."* — despite explicitly asking for
  "canonical, deduplicated," the agent spent 4 rounds and 8 tool calls across `semantic_search`,
  `fulltext_search`, `get_document_pages`, and `get_recognised_entities` (the raw, undeduplicated
  version) without ever calling `query_graph`, landing on a heavily-caveated answer ("cannot
  fully reconcile... detailed rollforward... not included").

Tool *availability* isn't the same as tool *usage*, and a working capability isn't the same as a
*reliably generated* one — the strategy LLM's docstring-driven selection is imperfect, and even
once it does reach for `query_graph`, a multi-hop join is measurably more fragile to generate
correctly than a single-hop one. Both are at least as important a limitation as anything in
section 7 below.

## 7. Honest tradeoffs — when NL→Cypher is/isn't trustworthy

What this notebook actually demonstrated, not the idealized pitch:

- **It closes a real gap.** Section 5's comparison is the clean case: a genuine cross-cutting
  aggregate that no fixed structured tool could answer, answered correctly in one round versus
  a text-only agent that correctly gave up rather than guess.
- **A syntactically valid, safety-clean query can still be quietly wrong — and better schema
  metadata measurably shifts the odds, without eliminating it.** Section 3 compared a naive
  schema representation against the real, `?`-flagged one on the exact same question: the
  naive version reproduces the `Company.name` bug noticeably more often, the flagged version
  shifts most runs to fully correct. But it's a shift in the odds from an LLM choosing to
  defend against a flagged nullable field, not an enforced guarantee — a minority of runs still
  lose data, and the schema-serialization layer still can't warn about a `null` that isn't
  structurally visible as one (a property present on every node but holding a wrong or stale
  *value* wouldn't trip anything). The validator can't catch any of this either — it's a
  read-only, syntactically correct query; validation is a write-safety gate, not a correctness
  gate. There is still no ground-truth check anywhere in this chain, by design — and, as
  section 3's closing discussion lays out, a correct-looking answer never proves it's actually
  correct.
- **The write guardrail doesn't depend on the LLM cooperating — and needed adversarial testing
  against itself, not just against the model.** Section 4 showed the model declining a
  destructive request on its own — reassuring, but `validate_cypher` is what actually stops a
  write, checked independently of what the model produces. Testing the validator itself
  adversarially (not just with the obvious `DELETE`/`DETACH` cases) found a real gap — a
  disguised `CREATE ... RETURN` slipping through the original pattern — since fixed, plus a
  second, still-open one: a write reached through a camelCase APOC procedure name
  (`apoc.refactor.mergeNodes`) that a word-boundary regex can't catch without also flagging
  legitimate reads. The honest conclusion isn't "the validator works," it's "a keyword-matching
  validator is a useful, fast first layer, not a substitute for a read-only database
  role/connection as the actual, unbypassable boundary."
- **`text2cypher.chain` itself has no self-correction loop — but the agent around it provides a
  coarser, emergent one.** `run_text_to_cypher` has no `try`/`except` around execution; a
  failed query (a `CypherSyntaxError`, a version-specific incompatibility, anything else)
  propagates uncaught, and nothing feeds the driver's error back to the model within one call
  (section 4). Section 6 showed the more complete picture: across *agent* retrieval rounds,
  each `query_graph` call is an independent generation, and the strategy LLM does see
  `grade_retrieval`'s natural-language feedback between rounds — which was enough, in that
  section's transcript, to steer consecutive invalid queries toward a correct one over a few
  rounds. That's real self-correction, but it happens at the agent-orchestration layer through
  plain-English retries, not inside the tool — slower, coarser, and unrelated to anything
  `chain.py` does on purpose.
- **A working capability isn't a reliably generated one.** Section 6's second query — joining
  `FinancialPeriod` with an `EntityGroup`-canonicalized traversal in a single call — measurably
  succeeded less often than any single-hop query in this notebook, and for a specific,
  repeatable reason (a wrong assumption about per-year scoping), not random noise. The deeper
  the join, the less you should trust any one generation to get it right on the first try.
- **Tool availability isn't tool usage.** Section 6 tried two other natural phrasings of the
  same underlying question that never reached `query_graph` at all — the strategy LLM's
  docstring-driven tool selection is itself an imperfect layer on top of everything else in
  this list.

Net: `query_graph` earns its place as a *last-resort* tool precisely because of these limits —
it's registered behind eight more reliable, narrower tools, and the strategy prompt
(`MODULE_6_STRATEGY_HINT`) tells the agent to reach for it only when nothing else fits. Section
6 showed the honest version of its upside too: real financial-domain questions — validating an
LLM-extracted number against its own duplicate mentions, cross-checking a qualitative
disclosure against vendor-verified financials — that no other tool in this agent can answer at
all, answered correctly, alongside the real cost of getting there. It is not a substitute for
the structured tools, and nothing in this module's design tries to make it one.